# 01 — Data Processing & Integration

**Goal**: Merge all raw CSVs into a single, clean ball-by-ball dataset (`final_processed_data.csv`).

**Input files** (all in `data/csv/`):
- `ballbyball.csv` — every delivery of every match
- `matches.csv` — match metadata (venue, time_of_day)
- `players.csv` — player profiles (batting/bowling style, playing role)
- `ground.csv`, `team.csv`, `season.csv`, `country.csv`, `town.csv` — reference tables

**Output**: `data/final_processed_data.csv`

**Pipeline**:
1. Load all CSVs
2. Merge ball-by-ball with match metadata → add `Ground Name`, `time_of_day`
3. Merge batsman player info → add `Full Name`, `Batsman_Batting_Style`, `Batsman_Playing_Role`
4. Merge bowler player info → add `Full Name_bowler`, `Bowler_Bowling_Style`, `Bowler_Playing_Role`
5. Drop irrelevant / redundant columns
6. Drop rows with nulls in key analytic columns
7. Correct totalRuns/totalWickets to pre-delivery values
8. Save clean dataset

In [1]:
import pandas as pd
import numpy as np
import os

# ── Path configuration ────────────────────────────────────────────────────────
# Set PROJECT_ROOT to the root folder of this project.
# Everything else is derived from it so you only ever change one line.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_RAW  = os.path.join(PROJECT_ROOT, 'data', 'csv')
DATA_OUT  = os.path.join(PROJECT_ROOT, 'data')

print('PROJECT_ROOT :', PROJECT_ROOT)
print('Raw data dir :', DATA_RAW)
print('Output dir   :', DATA_OUT)

PROJECT_ROOT : c:\Users\yaswa\OneDrive\Desktop\projects\artificial intelligence project
Raw data dir : c:\Users\yaswa\OneDrive\Desktop\projects\artificial intelligence project\data\csv
Output dir   : c:\Users\yaswa\OneDrive\Desktop\projects\artificial intelligence project\data


## 1. Load Raw Data

In [2]:
ball_df    = pd.read_csv(os.path.join(DATA_RAW, 'ballbyball.csv'))
matches_df = pd.read_csv(os.path.join(DATA_RAW, 'matches.csv'))
players_df = pd.read_csv(os.path.join(DATA_RAW, 'players.csv'))
ground_df  = pd.read_csv(os.path.join(DATA_RAW, 'ground.csv'))

print(f'ball_df    : {ball_df.shape}')
print(f'matches_df : {matches_df.shape}')
print(f'players_df : {players_df.shape}')
print(f'ground_df  : {ground_df.shape}')
ball_df.head(3)

ball_df    : (60326, 26)
matches_df : (296, 18)
players_df : (912, 12)
ground_df  : (36, 5)


,Unnamed: 0,match_id,match_obj_id,inningNumber,overNumber,ballNumber,oversUnique,oversActual,batsmanPlayerId,bowlerPlayerId,...,legbyes,wides,noballs,penalties,run,batsmanRuns,totalRuns,totalWickets,outPlayerId,shotType
0,0,100706,1273712,1,1,1,0.01,0.1,52584,87127,...,0,0,0,0,0,0,0,0,NaN,DEFENDED
1,1,100706,1273712,1,1,2,0.02,0.2,52584,87127,...,0,0,0,0,0,0,0,0,NaN,CUT_SHOT
2,2,100706,1273712,1,1,3,0.03,0.3,52584,87127,...,0,0,0,0,0,0,0,0,NaN,DEFENDED


## 2. Correct Pre-delivery Running Totals

Raw data stores **post-delivery** totals. For analysis we want the score/wickets *before* each delivery.

In [3]:
# Convert boolean columns to int first
for col in ['isFour', 'isSix', 'isWicket']:
    ball_df[col] = ball_df[col].astype(int)

# Subtract the current delivery's contribution to get pre-delivery values
ball_df['totalRuns']    = ball_df['totalRuns']    - ball_df['run']
ball_df['totalWickets'] = ball_df['totalWickets'] - ball_df['isWicket']

print('totalRuns range   :', ball_df['totalRuns'].min(), '–', ball_df['totalRuns'].max())
print('totalWickets range:', ball_df['totalWickets'].min(), '–', ball_df['totalWickets'].max())

totalRuns range   : 0 – 229
totalWickets range: 0 – 9


## 3. Merge Match Metadata

In [4]:
# Keep only essential match columns
ground_df  = ground_df[['Ground ID', 'Ground Name']]
matches_df = matches_df.rename(columns={'ground_id': 'Ground ID'})
matches_df = matches_df.merge(ground_df, on='Ground ID', how='left')
matches_keep = ['match_id', 'time_of_day', 'Ground Name']
matches_df = matches_df[matches_keep]

ball_df = ball_df.merge(matches_df, on='match_id', how='inner')
print('After match merge:', ball_df.shape)

After match merge: (60326, 28)


## 4. Drop Unnecessary Ball-level Columns

In [5]:
drop_cols = ['match_id', 'overNumber', 'ballNumber', 'oversUnique',
             'penalties', 'batsmanRuns', 'outPlayerId']
ball_df.drop(columns=[c for c in drop_cols if c in ball_df.columns], inplace=True)
print('After dropping ball columns:', ball_df.shape)

After dropping ball columns: (60326, 21)


## 5. Merge Player Information

In [6]:
# Clean up players table — keep only what we need
players_df = players_df[['ID', 'Name', 'Role', 'Full Name',
                          'Batting Style (l)', 'Bowling Style(l)', 'Playing Role']]

# ── Batsman info ──────────────────────────────────────────────────────────────
ball_df = ball_df.merge(
    players_df.rename(columns={
        'Name'           : 'Batsman_Name',
        'Role'           : 'Batsman_Role',
        'Batting Style (l)': 'Batsman_Batting_Style',
        'Bowling Style(l)' : 'Batsman_Bowling_Style',  # will drop later
        'Playing Role'   : 'Batsman_Playing_Role'
    }),
    left_on='batsmanPlayerId', right_on='ID', how='left'
).drop(columns=['ID'])

# ── Bowler info ───────────────────────────────────────────────────────────────
ball_df = ball_df.merge(
    players_df.rename(columns={
        'Name'           : 'Bowler_Name',
        'Role'           : 'Bowler_Role',
        'Full Name'      : 'Full Name_bowler',
        'Batting Style (l)': 'Bowler_Batting_Style',  # will drop later
        'Bowling Style(l)' : 'Bowler_Bowling_Style',
        'Playing Role'   : 'Bowler_Playing_Role'
    }),
    left_on='bowlerPlayerId', right_on='ID', how='left', suffixes=('', '_bowler')
).drop(columns=['ID'])

# Drop foreign keys and redundant style columns
ball_df.drop(columns=['batsmanPlayerId', 'bowlerPlayerId',
                       'Batsman_Bowling_Style', 'Bowler_Batting_Style',
                       'Batting Style (s)_bowler'],
             inplace=True, errors='ignore')

print('After player merge:', ball_df.shape)
ball_df.head(3)

After player merge: (60326, 29)


,Unnamed: 0,match_obj_id,inningNumber,oversActual,pitchLine,pitchLength,isFour,isSix,isWicket,byes,...,Batsman_Name,Batsman_Role,Full Name,Batsman_Batting_Style,Batsman_Playing_Role,Bowler_Name,Bowler_Role,Full Name_bowler,Bowler_Bowling_Style,Bowler_Playing_Role
0,0,1273712,1,0.1,ON_THE_STUMPS,GOOD_LENGTH,0,0,0,0,...,TP Ura,P,Tony Ura,right-hand bat,opening batter,Bilal Khan,P,Bilal Khan,left-arm medium-fast,bowler
1,1,1273712,1,0.2,ON_THE_STUMPS,SHORT_OF_A_GOOD_LENGTH,0,0,0,0,...,TP Ura,P,Tony Ura,right-hand bat,opening batter,Bilal Khan,P,Bilal Khan,left-arm medium-fast,bowler
2,2,1273712,1,0.3,ON_THE_STUMPS,GOOD_LENGTH,0,0,0,0,...,TP Ura,P,Tony Ura,right-hand bat,opening batter,Bilal Khan,P,Bilal Khan,left-arm medium-fast,bowler


## 6. Drop Rows with Missing Analytical Fields

In [7]:
required_cols = [
    'pitchLine', 'pitchLength', 'shotType',
    'Batsman_Name', 'Batsman_Batting_Style', 'Batsman_Playing_Role',
    'Bowler_Name',  'Bowler_Bowling_Style',  'Bowler_Playing_Role'
]

before = len(ball_df)
df = ball_df.dropna(subset=required_cols).copy()
df.reset_index(drop=True, inplace=True)
print(f'Rows kept: {len(df):,}  (dropped {before - len(df):,} rows with missing values)')

Rows kept: 33,029  (dropped 27,297 rows with missing values)


## 7. Final Column Order & Types

In [8]:
# Standardise boolean columns to int (0/1)
for col in ['isFour', 'isSix', 'isWicket']:
    df[col] = df[col].astype(int)

df['run'] = pd.to_numeric(df['run'], errors='coerce').fillna(0).astype(int)

# Select and order final columns
final_cols = [
    'match_obj_id', 'inningNumber', 'oversActual',
    'pitchLine', 'pitchLength', 'shotType',
    'isFour', 'isSix', 'isWicket',
    'byes', 'legbyes', 'wides', 'noballs',
    'run', 'totalRuns', 'totalWickets',
    'time_of_day', 'Ground Name',
    'Batsman_Name',   'Full Name',        'Batsman_Batting_Style', 'Batsman_Playing_Role',
    'Bowler_Name',    'Full Name_bowler', 'Bowler_Bowling_Style',  'Bowler_Playing_Role'
]
df = df[[c for c in final_cols if c in df.columns]]

print('Final shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(3)

Final shape: (33029, 26)
Columns: ['match_obj_id', 'inningNumber', 'oversActual', 'pitchLine', 'pitchLength', 'shotType', 'isFour', 'isSix', 'isWicket', 'byes', 'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets', 'time_of_day', 'Ground Name', 'Batsman_Name', 'Full Name', 'Batsman_Batting_Style', 'Batsman_Playing_Role', 'Bowler_Name', 'Full Name_bowler', 'Bowler_Bowling_Style', 'Bowler_Playing_Role']


,match_obj_id,inningNumber,oversActual,pitchLine,pitchLength,shotType,isFour,isSix,isWicket,byes,...,time_of_day,Ground Name,Batsman_Name,Full Name,Batsman_Batting_Style,Batsman_Playing_Role,Bowler_Name,Full Name_bowler,Bowler_Bowling_Style,Bowler_Playing_Role
0,1273712,1,0.1,ON_THE_STUMPS,GOOD_LENGTH,DEFENDED,0,0,0,0,...,day,Al Amerat Cricket Ground Oman Cricket (Ministr...,TP Ura,Tony Ura,right-hand bat,opening batter,Bilal Khan,Bilal Khan,left-arm medium-fast,bowler
1,1273712,1,0.2,ON_THE_STUMPS,SHORT_OF_A_GOOD_LENGTH,CUT_SHOT,0,0,0,0,...,day,Al Amerat Cricket Ground Oman Cricket (Ministr...,TP Ura,Tony Ura,right-hand bat,opening batter,Bilal Khan,Bilal Khan,left-arm medium-fast,bowler
2,1273712,1,0.3,ON_THE_STUMPS,GOOD_LENGTH,DEFENDED,0,0,0,0,...,day,Al Amerat Cricket Ground Oman Cricket (Ministr...,TP Ura,Tony Ura,right-hand bat,opening batter,Bilal Khan,Bilal Khan,left-arm medium-fast,bowler


## 8. Validation Checks

In [9]:
print('=== Data Quality Report ===')
print(f'Total deliveries     : {len(df):,}')
print(f'Unique batsmen       : {df["Full Name"].nunique()}')
print(f'Unique bowlers       : {df["Bowler_Name"].nunique()}')
print(f'Unique matches       : {df["match_obj_id"].nunique()}')
print(f'Missing values total : {df.isnull().sum().sum()}')
print()
print('Boundary rate  :', round((df['isFour'] | df['isSix']).mean() * 100, 1), '%')
print('Wicket rate    :', round(df['isWicket'].mean() * 100, 1), '%')
print('Dot ball rate  :', round((df['run'] == 0).mean() * 100, 1), '%')
print()
print('pitchLine distribution:')
print(df['pitchLine'].value_counts())
print()
print('pitchLength distribution:')
print(df['pitchLength'].value_counts())

=== Data Quality Report ===
Total deliveries     : 33,029
Unique batsmen       : 462
Unique bowlers       : 318
Unique matches       : 149
Missing values total : 0

Boundary rate  : 13.1 %
Wicket rate    : 5.4 %
Dot ball rate  : 38.0 %

pitchLine distribution:
pitchLine
OUTSIDE_OFFSTUMP         16902
ON_THE_STUMPS            11705
DOWN_LEG                  2356
WIDE_OUTSIDE_OFFSTUMP     1920
WIDE_DOWN_LEG              146
Name: count, dtype: int64

pitchLength distribution:
pitchLength
GOOD_LENGTH               14524
SHORT_OF_A_GOOD_LENGTH     7763
FULL                       6737
SHORT                      1996
FULL_TOSS                  1153
YORKER                      856
Name: count, dtype: int64


## 9. Save Output

In [10]:
out_path = os.path.join(DATA_OUT, 'final_processed_data.csv')
df.to_csv(out_path, index=False)
print(f'Saved → {out_path}  ({len(df):,} rows × {len(df.columns)} cols)')

Saved → c:\Users\yaswa\OneDrive\Desktop\projects\artificial intelligence project\data\final_processed_data.csv  (33,029 rows × 26 cols)
